# Lab 07-06 · AC Power and Power Factor

Companion to chapter [07-06](../../chapters/07-06-ac-power-and-power-factor.md). Handout: [README.md](README.md).

Every power below is computed from sampled waveforms ($P$ = the average of $v(t)i(t)$, RMS = the square root of the average square), so the phasor formulas of the chapter are being checked, not assumed. Cells marked **🔮 Predict** ask for a prediction *before* running.

In [1]:
import cmath
import math
import shutil
import subprocess
from pathlib import Path
import numpy as np
from scipy.optimize import brentq, minimize_scalar
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

HAVE_NGSPICE = shutil.which("ngspice") is not None
print("ngspice available:", HAVE_NGSPICE)
rms = lambda x: float(np.sqrt(np.mean(np.asarray(x) ** 2)))


def ngspice_values(path):
    out = subprocess.run(["ngspice", "-b", path], capture_output=True, text=True).stdout
    vals = {}
    for line in out.splitlines():
        parts = line.split("=")
        if len(parts) >= 2:
            try:
                vals[parts[0].strip()] = float(parts[1].split()[0])
            except (ValueError, IndexError):
                pass
    return vals


NG = ngspice_values("power.cir") if HAVE_NGSPICE else {}

ngspice available: True


## Part 1 — Power in waveforms

**🔮 Predict:** for a load at power factor 0.5, what fraction of each cycle is the instantaneous power negative?

In [2]:
t = np.linspace(0, 1, 100000, endpoint=False)          # one period, normalized time
fig, ax = plt.subplots(figsize=(6.4, 3.2))
for theta in (0, 37.02, 60, 90):
    p = np.cos(2 * np.pi * t) * np.cos(2 * np.pi * t - np.radians(theta))
    frac = np.mean(p < 0)
    ax.plot(t, p, label=f"θ = {theta}°: average {p.mean():.3f}, negative {frac * 100:.1f}% of the time")
    assert abs(frac - theta / 180) < 1e-3
ax.axhline(0, color="0.5", lw=0.6)
ax.set_xlabel("time / period"); ax.set_ylabel("p / (Vm Im)"); ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
plt.show()

The power is negative for exactly a fraction θ/180° of each cycle: 0 for a resistor, 20.6% for Example 07-06.1's load (θ = 37.02°), a third at PF 0.5 (θ = 60°), half for a pure reactance. The average falls as $\cos\theta/2$.

## Part 2 — RMS by quadrature

**🔮 Predict:** the RMS of $10\cos\omega t + 3\cos3\omega t + 1\cos5\omega t$.

In [3]:
x = 10 * np.cos(2 * np.pi * t) + 3 * np.cos(6 * np.pi * t) + np.cos(10 * np.pi * t)
quad = math.sqrt((10**2 + 3**2 + 1**2) / 2)
print(f"sampled RMS {rms(x):.4f}, quadrature rule {quad:.4f}, arithmetic sum of RMS values {(10 + 3 + 1) / math.sqrt(2):.4f}")
assert abs(rms(x) - quad) < 1e-9

sampled RMS 7.4162, quadrature rule 7.4162, arithmetic sum of RMS values 9.8995


## Part 3 — Correction and balance

Example 07-06.3's loads at 230 V, 50 Hz, with a parallel capacitor $C$.

**🔮 Predict:** how does the source current change as $C$ grows from 0 past the unity-power-factor value?

In [4]:
V, f = 230.0, 50.0
w = 2 * np.pi * f
ZA = V**2 / (10e3 - 1j * 10e3 * math.tan(math.acos(0.75)))
ZB = V**2 / 4e3
tt = np.linspace(0, 1 / f, 20000, endpoint=False)
vt = math.sqrt(2) * V * np.cos(w * tt)


def i_source(C):
    I = math.sqrt(2) * V * (1 / ZA + 1 / ZB + 1j * w * C)
    return abs(I) * np.cos(w * tt + cmath.phase(I))


Cs = np.linspace(0, 1e-3, 401)
Irms = [rms(i_source(C)) for C in Cs]
PF = [float(np.mean(vt * i_source(C))) / (V * rms(i_source(C))) for C in Cs]
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.plot(Cs * 1e6, Irms, label="source current (A rms)")
ax2 = ax.twinx(); ax2.plot(Cs * 1e6, PF, color="C1", ls="--", label="power factor"); ax2.set_ylim(0.5, 1.02)
ax.set_xlabel("capacitance (µF)"); ax.set_ylabel("A"); ax.grid(True, alpha=0.3)
plt.show()
Cu = minimize_scalar(lambda C: rms(i_source(C)), bounds=(0, 1e-3), method="bounded", options={"xatol": 1e-12}).x
print(f"minimum current {rms(i_source(Cu)):.2f} A at C = {Cu * 1e6:.1f} uF (unity PF)")
# conservation of complex power, element by element, at C = 253.8 uF
C = 253.78e-6
Vp = math.sqrt(2) * V
S = {"motor": 0.5 * Vp * np.conj(Vp / ZA), "heating": 0.5 * Vp * np.conj(Vp / ZB), "capacitor": 0.5 * Vp * np.conj(Vp * 1j * w * C)}
S_src = 0.5 * Vp * np.conj(Vp * (1 / ZA + 1 / ZB + 1j * w * C))
for k, s_ in S.items():
    print(f"{k:9s}: {s_.real:8.1f} W {s_.imag:+9.1f} var")
print(f"source   : {S_src.real:8.1f} W {S_src.imag:+9.1f} var (sum of loads {sum(S.values()).real:.1f} W {sum(S.values()).imag:+.1f} var)")
assert abs(S_src - sum(S.values())) < 1e-6
if NG:
    print("ngspice: current before", NG["ex3i0"], "A, after", NG["ex3i1"], "A")

minimum current 60.87 A at C = 530.7 uF (unity PF)
motor    :  10000.0 W   +8819.2 var
heating  :   4000.0 W      +0.0 var
capacitor:      0.0 W   -4217.6 var
source   :  14000.0 W   +4601.6 var (sum of loads 14000.0 W +4601.6 var)
ngspice: current before 71.9402 A, after 64.0732 A


The current is smallest at unity power factor (60.87 A at 530.7 µF) and rises again beyond it, as the circuit turns leading. Real and reactive power balance separately at every capacitor value.

## Part 4 — Rectifier pulse width

**🔮 Predict:** do wider pulses raise or lower the power factor at the same 100 W?

In [5]:
f, T = 50.0, 0.02
t = np.linspace(0, T, 200000, endpoint=False)
vt = 230 * math.sqrt(2) * np.cos(2 * np.pi * f * t)
rows = []
for W in np.array([1.0, 2.5, 4.0, 5.0, 7.0, 9.0]) * 1e-3:
    shape = ((np.abs(t) < W / 2) | (np.abs(t - T) < W / 2)).astype(float) - (np.abs(t - T / 2) < W / 2).astype(float)
    Ip = 100.0 / float(np.mean(vt * shape))
    i = Ip * shape
    F = np.fft.rfft(i) / (len(t) / 2)
    I1 = abs(F[1]) / math.sqrt(2)
    pf = 100 / (230 * rms(i))
    thd = math.sqrt(rms(i) ** 2 - I1**2) / I1
    rows.append((W, pf, thd))
    print(f"W = {W * 1e3:3.1f} ms: PF {pf:.4f}, THD {thd * 100:5.1f}%, check cos/sqrt(1+THD^2) = {1 / math.sqrt(1 + thd**2):.4f}")
    assert abs(pf - 1 / math.sqrt(1 + thd**2)) < 1e-6
assert abs([r[1] for r in rows if abs(r[0] - 5e-3) < 1e-9][0] - 0.9003) < 5e-5

W = 1.0 ms: PF 0.4454, THD 201.0%, check cos/sqrt(1+THD^2) = 0.4454
W = 2.5 ms: PF 0.6891, THD 105.2%, check cos/sqrt(1+THD^2) = 0.6891
W = 4.0 ms: PF 0.8367, THD  65.4%, check cos/sqrt(1+THD^2) = 0.8367
W = 5.0 ms: PF 0.9003, THD  48.3%, check cos/sqrt(1+THD^2) = 0.9003
W = 7.0 ms: PF 0.9588, THD  29.6%, check cos/sqrt(1+THD^2) = 0.9588
W = 9.0 ms: PF 0.9373, THD  37.2%, check cos/sqrt(1+THD^2) = 0.9373


Wider pulses look more like a sinusoid, so THD falls and power factor rises: 0.6891 at 2.5 ms, 0.9003 at 5 ms. In every case PF $= 1/\sqrt{1 + \text{THD}^2}$, because the displacement factor is 1.

## Your turn

1. A load has PF 0.5. For what fraction of each cycle does it return energy to the supply?
2. What is the RMS of $10\cos\omega t + 3\cos3\omega t + 1\cos5\omega t$?
3. What capacitance gives Example 07-06.3 unity power factor, and what current then flows?

<details><summary>Answers</summary>

1. θ = 60°, so **one third** of each cycle (θ/180°).
2. $\sqrt{(100 + 9 + 1)/2} = $ **7.416**.
3. $C = Q/(\omega V^2) = 8819/(314.16\times230^2) = $ **530.7 µF**; the current is then $14\,000/230 = $ **60.87 A**.

</details>

In [6]:
print(f"2) {math.sqrt(55):.3f}")
Cu = minimize_scalar(lambda C: rms(i_source(C)), bounds=(0, 1e-3), method="bounded", options={"xatol": 1e-12}).x
print(f"3) {Cu * 1e6:.1f} uF, {rms(i_source(Cu)):.2f} A")
assert abs(Cu * 1e6 - 530.7) < 0.1 and abs(rms(i_source(Cu)) - 60.87) < 0.005

2) 7.416
3) 530.7 uF, 60.87 A
